# 04 - Model Training: CNN Transfer Learning & Experimental Design
**Diabetic Retinopathy Stage Detection**

Rubric criteria 4 (architecture + transfer learning, 20 marks) and 5 (training strategy, 10 marks).

Experimental design - every question answered with a controlled comparison on the **validation** set
(the test set is touched only once, in notebook 05):

| # | Question | Runs |
|---|---|---|
| E0 | Is transfer learning needed at all? | small CNN from scratch |
| E1 | Which imbalance remedy works best? | EfficientNet-B0 x {none, class weights, sqrt oversampling} |
| E2 | Which backbone? | {EfficientNet-B0, ResNet50-V2, DenseNet-121} with the E1 winner |
| E3 | Hyper-parameters | fine-tune LR and dropout sweep on the E2 winner |
| E4 | Final model | E2/E3 winner, long two-phase schedule with early stopping |

Training strategy (all runs): two-phase transfer learning (frozen backbone -> partial unfreeze with
BatchNorm frozen), Adam, on-the-fly augmentation, `ModelCheckpoint` on val QWK, `EarlyStopping`,
`ReduceLROnPlateau`, CSV logging, mixed-precision on GPU. All logic in `src/model.py` and `src/train.py`.

In [ ]:
# --- Environment bootstrap: works on Kaggle, Ubuntu GPU box and Windows ----
import os, sys, subprocess
from pathlib import Path

REPO = "https://github.com/pradeesha999/dr-stage-detection-v2.git"
if Path("/kaggle/working").exists():
    dst = Path("/kaggle/working/dr_project")
    if not dst.exists():
        try:   # private repo: store a GitHub token as Kaggle secret GITHUB_TOKEN
            from kaggle_secrets import UserSecretsClient
            url = REPO.replace("https://", f"https://{UserSecretsClient().get_secret('GITHUB_TOKEN')}@")
        except Exception:
            url = REPO                      # works as-is if the repo is public
        subprocess.run(["git", "clone", "-q", url, str(dst)], check=True)
        subprocess.run(["git", "-C", str(dst), "remote", "set-url", "origin", REPO])  # keep token out of .git/config
    os.chdir(dst / "notebooks")
sys.path.insert(0, str(Path.cwd().parent))     # notebooks/ -> dr_project/
os.environ.setdefault("TF_CPP_MIN_LOG_LEVEL", "2")

import json, time
import numpy as np, pandas as pd
import matplotlib.pyplot as plt, seaborn as sns
import tensorflow as tf, keras

from src import config, data, utils, dataset, train
from src import model as M
utils.set_seed()
sns.set_theme(style="whitegrid")

gpus = tf.config.list_physical_devices("GPU")
print("TF", tf.__version__, "| Keras", keras.__version__, "| GPUs:", gpus)
if gpus:
    keras.mixed_precision.set_global_policy("mixed_float16")   # ~2x faster on T4 tensor cores
    print("mixed precision ON")

train_df, val_df, test_df = data.load_splits()
assert Path(train_df.proc_path.iloc[0]).exists(), "run 02_preprocessing first (processed cache missing)"
print(len(train_df), len(val_df), len(test_df))

# QUICK=True -> tiny subset + 1 epoch per phase, just to check the notebook runs end-to-end.
QUICK = os.environ.get("DR_QUICK", "0") == "1"
if QUICK:
    train_df = train_df.groupby("label").head(60); val_df = val_df.groupby("label").head(30)
E = (lambda n: 1) if QUICK else (lambda n: n)      # epoch scaler
print("QUICK mode:", QUICK)

## E0 - Baseline: small CNN trained from scratch

In [ ]:
r = train.run_experiment("E0_baseline_cnn", train_df, val_df, baseline_cnn=True,
                         balancing="class_weights", epochs_head=E(8), epochs_finetune=0,
                         lr_head=1e-3, patience=4, notes="from scratch, no pretrained weights")

## E1 - Class-imbalance strategy (EfficientNet-B0)
Same backbone, same schedule, only the balancing changes. Compared on **macro-F1** and **QWK**
(accuracy is dominated by No_DR and would reward ignoring the minority stages).

In [ ]:
for bal in ["none", "class_weights", "sqrt"]:
    train.run_experiment(f"E1_effb0_{bal}", train_df, val_df, backbone="efficientnetb0",
                         balancing=bal, epochs_head=E(3), epochs_finetune=E(4), patience=3,
                         notes="imbalance ablation")
e1 = train.results_table(); e1 = e1[e1.run.str.startswith("E1")]
display(e1)
BEST_BAL = e1.iloc[0].balancing
print("-> best balancing:", BEST_BAL)

## E2 - Backbone comparison (with the best balancing strategy)

In [ ]:
for bb in ["efficientnetb0", "resnet50v2", "densenet121"]:
    train.run_experiment(f"E2_{bb}", train_df, val_df, backbone=bb, balancing=BEST_BAL,
                         epochs_head=E(3), epochs_finetune=E(5), patience=3, notes="backbone comparison")
e2 = train.results_table(); e2 = e2[e2.run.str.startswith("E2")]
display(e2)
BEST_BB = e2.iloc[0].backbone
print("-> best backbone:", BEST_BB)

## E3 - Hyper-parameter tuning on the winner (fine-tune LR x dropout)

In [ ]:
for lr_ft, dr in [(3e-5, 0.3), (1e-4, 0.5), (3e-4, 0.3)]:
    train.run_experiment(f"E3_{BEST_BB}_lr{lr_ft:g}_do{dr}", train_df, val_df, backbone=BEST_BB,
                         balancing=BEST_BAL, epochs_head=E(2), epochs_finetune=E(4), patience=3,
                         lr_finetune=lr_ft, dropout=dr, notes="hparam sweep")
e3 = train.results_table(); e3 = e3[e3.run.str.startswith(("E2_" + BEST_BB, "E3"))]
display(e3)
best = e3.iloc[0]
cfg = json.loads((train.METRIC_DIR / f"{best.run}.json").read_text())
LR_FT, DROPOUT = cfg["lr_finetune"], cfg["dropout"]
print(f"-> best fine-tune lr={LR_FT}, dropout={DROPOUT}")

## E4 - Final model: long two-phase schedule with early stopping
Head for 5 epochs at 1e-3, then top 30 % of the backbone unfrozen for up to 25 epochs
(early stopping, patience 6, on validation QWK). Label smoothing 0.1 for calibration.

In [ ]:
final = train.run_experiment("E4_final", train_df, val_df, backbone=BEST_BB, balancing=BEST_BAL,
                             epochs_head=E(5), epochs_finetune=E(25), patience=6,
                             lr_finetune=LR_FT, dropout=DROPOUT, label_smoothing=0.1,
                             notes="final model")
(train.METRIC_DIR / "final_run.txt").write_text("E4_final")

## Results: all experiments

In [ ]:
tbl = train.results_table()
tbl.to_csv(train.METRIC_DIR / "summary.csv", index=False)
display(tbl)

fig, ax = plt.subplots(figsize=(11, 4.5))
t = tbl.sort_values("run")
x = np.arange(len(t)); w = 0.27
ax.bar(x - w, t.val_accuracy, w, label="accuracy"); ax.bar(x, t.val_macro_f1, w, label="macro-F1"); ax.bar(x + w, t.val_qwk, w, label="QWK")
ax.set_xticks(x); ax.set_xticklabels(t.run, rotation=35, ha="right", fontsize=8); ax.set_ylim(0, 1); ax.legend()
ax.set_title("Validation metrics per experiment"); plt.tight_layout(); utils.save_fig("04_experiment_summary"); plt.show()

## Training curves (accuracy & loss) for the final model and the ablations

In [ ]:
def plot_curves(run, ax_pair, title=None):
    h = train.load_curves(run)
    a1, a2 = ax_pair
    a1.plot(h.accuracy, label="train"); a1.plot(h.val_accuracy, label="val"); a1.set_title(f"{title or run} - accuracy"); a1.legend()
    a2.plot(h.loss, label="train"); a2.plot(h.val_loss, label="val"); a2.set_title(f"{title or run} - loss"); a2.legend()
    cfg = json.loads((train.METRIC_DIR / f"{run}.json").read_text())
    if cfg["epochs_head"] and cfg["epochs_finetune"]:
        for a in (a1, a2): a.axvline(cfg["epochs_head"] - 0.5, ls="--", c="gray", lw=1)
    a1.set_xlabel("epoch"); a2.set_xlabel("epoch")

fig, ax = plt.subplots(1, 2, figsize=(12, 4)); plot_curves("E4_final", ax, "final model")
plt.tight_layout(); utils.save_fig("04_final_curves"); plt.show()

# QWK / macro-F1 over epochs for the final model
h = train.load_curves("E4_final")
plt.figure(figsize=(6, 4)); plt.plot(h.val_qwk, label="val QWK"); plt.plot(h.val_macro_f1, label="val macro-F1")
plt.axvline(final["epochs_head"] - 0.5, ls="--", c="gray", lw=1); plt.xlabel("epoch"); plt.legend(); plt.title("Final model - validation QWK / macro-F1")
plt.tight_layout(); utils.save_fig("04_final_qwk_f1"); plt.show()

runs = [r for r in tbl.run if r.startswith(("E0", "E1", "E2"))]
fig, ax = plt.subplots(len(runs), 2, figsize=(12, 3.4 * len(runs)))
for i, r in enumerate(runs): plot_curves(r, ax[i])
plt.tight_layout(); utils.save_fig("04_ablation_curves"); plt.show()

## Notes for the report
* Dashed vertical line in the curves = switch from phase 1 (frozen backbone) to phase 2 (fine-tuning).
* `outputs/metrics/summary.csv` is the experiment table; `outputs/models/E4_final.keras` is the model
  evaluated in notebook 05.
* Everything is seeded (`SEED=42`), so re-running reproduces the numbers up to GPU non-determinism.